# Aula 02 — Engenharia de Prompt na prática

**Disciplina:** Deploy de Agentes com LLMs — Pós-Graduação em Machine Learning (FIAP/POSTECH)

Nesta oficina partimos de **prompts ruins** e os melhoramos passo a passo, observando como
cada técnica de *prompt engineering* muda a saída do modelo. No final, montamos um agente
no padrão **ReAct** com uma ferramenta de **busca**.

**Princípios que vamos exercitar:**
1. Clareza e especificidade
2. Papel e tom (mensagem de *system*)
3. Delimitadores e saída estruturada (JSON)
4. Exemplos (*few-shot*)
5. Raciocínio passo a passo (*Chain-of-Thought*)
6. Aterramento e combate à alucinação
7. **Bônus:** o padrão **ReAct** (raciocínio + ação com ferramentas)

> **Requer uma chave de API.** Configure o `.env` na pasta `aula-02/` antes de executar.
> Padrão: **Claude** (`ANTHROPIC_API_KEY`); use `LLM_PROVIDER=openai` (+ `OPENAI_API_KEY`) para a OpenAI.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

# Carrega as variáveis do .env (procura aula-02/.env a partir da pasta notebooks/).
load_dotenv(find_dotenv(usecwd=True))


def get_modelo(temperature=0.0):
    """Cria o chat model (Claude por padrão, ou OpenAI via LLM_PROVIDER=openai)."""
    prov = os.getenv("LLM_PROVIDER", "anthropic").lower()
    chave = "ANTHROPIC_API_KEY" if prov == "anthropic" else "OPENAI_API_KEY"
    if not os.getenv(chave):
        raise RuntimeError(
            f"Defina {chave} no arquivo .env (pasta aula-02/) para executar o notebook.")
    # `or` cobre o caso de LLM_MODEL estar vazio no .env.
    if prov == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=os.getenv("LLM_MODEL") or "claude-sonnet-4-6", temperature=temperature)
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model=os.getenv("LLM_MODEL") or "gpt-4o-mini", temperature=temperature)


def responder(mensagens, temperature=0.0, stop=None):
    """Chama o LLM e devolve o texto da resposta."""
    modelo = get_modelo(temperature)
    if stop:
        modelo = modelo.bind(stop=stop)
    return modelo.invoke(mensagens).content


def comparar(rotulo_ruim, saida_ruim, rotulo_bom, saida_bom):
    """Imprime lado a lado o prompt fraco e o prompt forte."""
    print("=" * 72)
    print("PROMPT FRACO -> ", rotulo_ruim)
    print("-" * 72)
    print(saida_ruim.strip())
    print("\n" + "=" * 72)
    print("PROMPT FORTE -> ", rotulo_bom)
    print("-" * 72)
    print(saida_bom.strip())


prov = os.getenv("LLM_PROVIDER", "anthropic").lower()
modelo_nome = os.getenv("LLM_MODEL") or ("claude-sonnet-4-6" if prov == "anthropic" else "gpt-4o-mini")
get_modelo()  # valida que a chave está configurada (lança erro claro se faltar)
print(f"Ambiente pronto. Provedor: {prov} | Modelo: {modelo_nome}")

Ambiente pronto. Provedor: anthropic | Modelo: claude-sonnet-4-6


## 1. Clareza e especificidade

O modelo não adivinha o que você quer. Um pedido vago gera uma saída genérica; um pedido
**específico** — com público, fatos, tom e limite de tamanho — gera exatamente o que você
precisa.

In [2]:
# Tarefa: escrever a descrição de um produto para um e-commerce.

# Prompt vago: sem público, tamanho, tom ou fatos.
ruim = "Escreva sobre nosso fone de ouvido."
saida_ruim = responder([{"role": "user", "content": ruim}])

# Prompt específico: papel + fatos + público + tom + limite de tamanho.
bom = ("Você é redator de e-commerce. Escreva a descrição do fone SomTop Air "
       "(Bluetooth 5.3, 30h de bateria, cancelamento de ruído) em no máximo 40 palavras, "
       "tom animado, para um público jovem. Destaque a bateria.")
saida_bom = responder([{"role": "user", "content": bom}])

comparar(ruim, saida_ruim, bom, saida_bom)

PROMPT FRACO ->  Escreva sobre nosso fone de ouvido.
------------------------------------------------------------------------
# Fone de Ouvido — Qualidade que Você Sente

Preciso de mais informações para escrever sobre o seu fone de ouvido! 😊

Mas aqui estão algumas opções de texto que posso criar para você:

---

## O que você precisa?

| Tipo de Texto | Descrição |
|--------------|-----------|
| 📢 **Descrição de produto** | Para e-commerce ou catálogo |
| 📱 **Post para redes sociais** | Instagram, Facebook, etc. |
| 📰 **Artigo/Review** | Análise detalhada |
| 📣 **Slogan/Campanha** | Frase de impacto |
| 📧 **E-mail marketing** | Para clientes |

---

## Me conte mais sobre o produto:

- **Qual é o nome/modelo** do fone?
- **Quais são as características** principais? *(sem fio, cancelamento de ruído, etc.)*
- **Qual é o público-alvo?** *(gamers, músicos, uso casual)*
- **Qual é o diferencial** do produto?
- **Qual é o tom desejado?** *(técnico, descontraído, sofisticado)*

---

Com ess

## 2. Papel e tom — *system message*

O **system message** define *quem* o modelo é, *como* deve falar e *quais fatos* pode
usar. Sem ela, o modelo não sabe o contexto e responde de forma genérica ou pede mais dados.

In [3]:
pergunta = "A bateria dura quanto?"

# Sem system: o modelo não conhece o papel nem os fatos do produto.
saida_ruim = responder([{"role": "user", "content": pergunta}])

# Com system: papel, tom e fatos que o modelo deve usar.
system = ("Você é a Ana, atendente da loja SomTop. Seja simpática e objetiva. "
          "Fatos do fone SomTop Air: 30h de bateria; carga rápida de 10 min rende 5h. "
          "Responda apenas com base nesses fatos.")
saida_bom = responder([{"role": "system", "content": system},
                       {"role": "user", "content": pergunta}])

comparar("(sem system) " + pergunta, saida_ruim, "(com system) " + pergunta, saida_bom)

PROMPT FRACO ->  (sem system) A bateria dura quanto?
------------------------------------------------------------------------
Poderia ser mais específico? **"A bateria"** pode se referir a muitas coisas, por exemplo:

- 📱 **Celular/Smartphone**
- 💻 **Notebook/Laptop**
- 🚗 **Carro**
- 🎧 **Fone de ouvido**
- ⌚ **Smartwatch**
- 🔋 **Outro dispositivo**

Me diga **qual produto ou dispositivo** você está perguntando para eu te dar uma resposta mais precisa! 😊

PROMPT FORTE ->  (com system) A bateria dura quanto?
------------------------------------------------------------------------
Olá! 😊 A bateria do **SomTop Air** dura **30 horas** de uso! E o melhor: com apenas **10 minutinhos de carga rápida**, você ganha mais **5 horas** de música. Super prático, né? 🎧

Posso te ajudar com mais alguma informação?


## 3. Delimitadores e saída estruturada

Para usar a resposta **dentro de uma aplicação**, peça um formato fixo (ex.: JSON) e isole o
texto de entrada com **delimitadores** (`<<< >>>`, crases, tags). Isso evita ambiguidade e
torna a saída fácil de processar.

In [4]:
review = ("Comprei o fone semana passada. O som é excelente e a bateria dura muito, "
          "mas o estojo chegou riscado. Recomendo!")

# Sem estrutura: saída em prosa, difícil de consumir em código.
saida_ruim = responder([{"role": "user", "content": "Extraia as informações deste review: " + review}])

# Delimitadores isolam o texto + schema JSON fixo.
prompt = ("Extraia os dados do review delimitado por <<< >>> e responda APENAS com um JSON "
          "com as chaves: sentimento (positivo/negativo/neutro), pontos_positivos (lista), "
          "pontos_negativos (lista), recomenda (booleano).\n\n<<<" + review + ">>>")
saida_bom = responder([{"role": "user", "content": prompt}])

comparar("(texto livre)", saida_ruim, "(delimitador + JSON)", saida_bom)

PROMPT FRACO ->  (texto livre)
------------------------------------------------------------------------
## Extração de Informações do Review

| Campo | Informação |
|---|---|
| **Produto** | Fone de ouvido |
| **Data da Compra** | Semana passada |
| **Avaliação Geral** | Positiva ✅ |
| **Recomenda?** | Sim ✅ |

---

### 👍 Pontos Positivos
- Qualidade de som **excelente**
- Bateria com **longa duração**

### 👎 Pontos Negativos
- Estojo chegou **riscado** (problema na entrega/embalagem)

---

> **Resumo:** Cliente satisfeito com o produto em si, mas insatisfeito com o estado físico do acessório (estojo) no momento da entrega.

PROMPT FORTE ->  (delimitador + JSON)
------------------------------------------------------------------------
```json
{
  "sentimento": "positivo",
  "pontos_positivos": ["som excelente", "bateria de longa duração"],
  "pontos_negativos": ["estojo chegou riscado"],
  "recomenda": true
}
```


### Structured output — deixe o framework garantir o JSON

Pedir JSON no prompt funciona, mas o modelo pode escorregar: texto antes/depois do JSON,
vírgula sobrando, uma chave faltando. O recurso de **structured output** elimina esse risco:
você declara o **schema** (aqui com *Pydantic*) e o framework **obriga** o modelo a devolvê-lo,
já **validado** e pronto como objeto Python — sem `json.loads` na mão.

> Por baixo dos panos, o LangChain usa *tool calling* (chamada de ferramenta) do modelo para
> forçar o formato. Funciona tanto com **Claude** quanto com **OpenAI**.

In [5]:
from pydantic import BaseModel, Field
from typing import Literal


# Declaramos o schema desejado como uma classe Pydantic (com tipos e descrições).
class AnaliseReview(BaseModel):
    """Estrutura validada para a análise de um review de produto."""
    sentimento: Literal["positivo", "negativo", "neutro"] = Field(
        description="Sentimento geral do cliente")
    pontos_positivos: list[str] = Field(description="Elogios citados no review")
    pontos_negativos: list[str] = Field(description="Reclamações citadas no review")
    recomenda: bool = Field(description="Se o cliente recomenda o produto")


# with_structured_output OBRIGA o modelo a devolver exatamente esse schema (via tool use),
# já validado pelo Pydantic — sem prompt pedindo "responda em JSON" e sem json.loads na mão.
modelo_estruturado = get_modelo().with_structured_output(AnaliseReview)
analise = modelo_estruturado.invoke([{"role": "user", "content": "Analise o review: " + review}])

print("Tipo retornado:", type(analise).__name__, "(objeto Python, não texto)")
print(analise)
print("\nAcesso direto por atributo (sem parse):")
print("  sentimento:", analise.sentimento)
print("  recomenda :", analise.recomenda)
print("  positivos :", analise.pontos_positivos)

Tipo retornado: AnaliseReview (objeto Python, não texto)
sentimento='positivo' pontos_positivos=['Som excelente', 'Bateria de longa duração'] pontos_negativos=['Estojo chegou riscado'] recomenda=True

Acesso direto por atributo (sem parse):
  sentimento: positivo
  recomenda : True
  positivos : ['Som excelente', 'Bateria de longa duração']


## 4. Exemplos (*few-shot*)

Quando o critério é subjetivo (o que é "urgente"?), descrevê-lo em palavras raramente basta.
Mostrar **alguns exemplos** de entrada→saída ancora tanto o critério quanto o formato da
resposta.

In [6]:
instrucao = "Classifique a urgência da mensagem de suporte como: baixa, média ou alta."
nova = 'Mensagem: "Estou sem acesso à conta e tenho uma entrega hoje!"'

# Zero-shot: o critério de "urgência" fica ambíguo.
saida_ruim = responder([{"role": "user", "content": instrucao + "\n" + nova}])

# Few-shot: exemplos ancoram o critério e o formato (só o rótulo).
mensagens = [
    {"role": "system", "content": instrucao + " Responda apenas com o rótulo."},
    {"role": "user", "content": 'Mensagem: "Vocês têm na cor azul?"'},
    {"role": "assistant", "content": "baixa"},
    {"role": "user", "content": 'Mensagem: "Fui cobrado duas vezes na fatura!"'},
    {"role": "assistant", "content": "alta"},
    {"role": "user", "content": 'Mensagem: "O produto chegou, mas veio sem o manual."'},
    {"role": "assistant", "content": "média"},
    {"role": "user", "content": nova},
]
saida_bom = responder(mensagens)

comparar("(zero-shot) " + nova, saida_ruim, "(few-shot) " + nova, saida_bom)

PROMPT FRACO ->  (zero-shot) Mensagem: "Estou sem acesso à conta e tenho uma entrega hoje!"
------------------------------------------------------------------------
## Classificação de Urgência: 🔴 **ALTA**

### Justificativa:

| Fator | Análise |
|-------|---------|
| ⏰ **Prazo imediato** | A entrega é **hoje**, sem margem de tempo |
| 🔒 **Bloqueio total** | Sem acesso à conta = impossibilidade de agir |
| 💼 **Impacto operacional** | Pode causar perda financeira ou contratual |
| 😰 **Tom da mensagem** | Indica estresse e urgência real |

### Ação Recomendada:
> Atendimento **imediato e prioritário**, com foco em restaurar o acesso o mais rápido possível para não comprometer a entrega do cliente.

PROMPT FORTE ->  (few-shot) Mensagem: "Estou sem acesso à conta e tenho uma entrega hoje!"
------------------------------------------------------------------------
alta


## 5. Raciocínio passo a passo (*Chain-of-Thought*)

Em tarefas de várias etapas (contas, lógica), pedir a resposta direta induz o modelo a
"chutar". Pedir para **pensar passo a passo** antes de concluir reduz muito os erros
(Wei et al., 2022).

In [7]:
problema = ("Um cliente comprou 3 fones a R$ 200 cada e usou um cupom de 15% de desconto "
            "no total. O frete foi R$ 25. Quanto ele pagou no final?")

# Resposta direta: sem raciocínio, para problemas complexos a possibilidade de erro é maior.
saida_ruim = responder([{"role": "user", "content": problema + " Responda apenas o valor."}])

# Chain-of-Thought: pedir o passo a passo antes de concluir.
saida_bom = responder([{"role": "user", "content": problema + " Pense passo a passo e só então conclua."}])

comparar("(resposta direta)", saida_ruim, "(chain-of-thought)", saida_bom)

PROMPT FRACO ->  (resposta direta)
------------------------------------------------------------------------
**R$ 535,00**

(3 × R$200 = R$600 → desconto 15% = R$90 → R$510 + R$25 de frete = **R$535,00**)

PROMPT FORTE ->  (chain-of-thought)
------------------------------------------------------------------------
## Resolução passo a passo

**1. Calcular o subtotal dos produtos:**
3 fones × R$ 200 = **R$ 600**

---

**2. Aplicar o desconto de 15%:**
15% de R$ 600 = 0,15 × 600 = **R$ 90 de desconto**

R$ 600 − R$ 90 = **R$ 510**

---

**3. Adicionar o frete:**
R$ 510 + R$ 25 = **R$ 535**

---

## ✅ Conclusão

O cliente pagou **R$ 535,00** no final.


## 6. Evitar alucinação usando contexto (in-context learning)

Quando a informação não está no prompt e nem no conhecimento do modelo (baseado em seu treinamento), 
o modelo cai em duas armadilhas: **inventar** um fato plausível (alucinação) 
ou dar uma resposta **genérica/hesitante** que não reflete o *seu*
negócio. A correção é a mesma: fornecer o contexto certo na própria
mensagem (*in-context*) e instruir o modelo a responder **somente** com base nele, admitindo
quando a informação não estiver lá.

No exemplo abaixo, a política real da SomTop dá **45 dias** para troca. Sem o contexto, o
modelo não sabe a política desta loja: responde de forma genérica (ex.: a regra geral do
Código de Defesa do Consumidor, 7 dias) ou pede para o cliente procurar outro canal. Com o
contexto, ele responde de forma direta e correta: **45 dias**.

> Modelos mais modernos tendem a hesitar em vez de inventar — mais seguros, mas
> ainda assim sem a resposta certa quando falta o contexto. Por isso, **dar contexto é essencial**.

In [8]:
# A política REAL da loja (fonte de verdade) — difere do padrão genérico do CDC (7 dias).
contexto = ("Política da loja SomTop:\n"
            "- Trocas e devoluções em até 45 dias após o recebimento.\n"
            "- Frete grátis acima de R$ 199.\n"
            "- Garantia de 12 meses contra defeitos.")
pergunta = "Quantos dias eu tenho para trocar um produto que comprei?"

# Sem o contexto, o modelo responde de forma generalista desconsiderando a política real desta loja.
saida_ruim = responder([{"role": "system", "content": "Você é atendente da loja SomTop."},
                        {"role": "user", "content": pergunta}])

# Responder só pelo contexto fornecido (a fonte de verdade da loja).
regra = ("Responda APENAS com base na POLÍTICA abaixo. Se a resposta não estiver nela, "
         'diga exatamente: "Não tenho essa informação."')
saida_bom = responder([{"role": "user", "content": regra + "\n\n" + contexto + "\n\nPergunta: " + pergunta}])

comparar("(sem contexto) " + pergunta, saida_ruim, "(com contexto) " + pergunta, saida_bom)

PROMPT FRACO ->  (sem contexto) Quantos dias eu tenho para trocar um produto que comprei?
------------------------------------------------------------------------
Olá! 😊

Para te dar uma informação precisa sobre o prazo de troca de produtos da **SomTop**, recomendo que você:

- **Entre em contato com nossa equipe** diretamente para verificar a política de trocas vigente.
- **Verifique o cupom fiscal ou nota fiscal** da sua compra, pois as condições podem estar descritas lá.
- **Acesse nosso site ou loja** para consultar a política oficial.

Posso te ajudar com mais alguma coisa? 😊

PROMPT FORTE ->  (com contexto) Quantos dias eu tenho para trocar um produto que comprei?
------------------------------------------------------------------------
Com base na política da **SomTop**, você tem **45 dias** após o recebimento do produto para realizar trocas e devoluções.


## Checklist de boas práticas

| Princípio | Pergunta-chave | Sinal de prompt fraco |
|-----------|----------------|------------------------|
| **Clareza** | Dei público, fatos, tom e tamanho? | Pedido genérico ("fale sobre…") |
| **Papel (system)** | Defini quem o modelo é e o que pode usar? | Modelo pede contexto ou divaga |
| **Estrutura** | Pedi um formato fixo e isolei a entrada? | Saída em prosa, difícil de parsear |
| **Few-shot** | Mostrei exemplos do critério/formato? | Rótulos inconsistentes |
| **Chain-of-Thought** | Pedi o passo a passo antes da resposta? | Erros em contas de várias etapas |
| **Aterramento** | Forneci contexto e proibi inventar? | Respostas plausíveis, porém falsas |

> Regra de ouro: **itere**. Comece simples, observe a falha e adicione *uma* técnica por vez.

> As saídas vêm de um LLM real, então variam de execução para execução (e com a `temperature`).
> Rode as células mais de uma vez para ver a diferença entre prompt fraco e forte de forma consistente.

## 7. Bônus — ReAct na prática com a Web Search nativa do Claude

Nas seções anteriores o modelo respondeu com base no próprio conhecimento. Mas e quando a
pergunta exige um **fato recente**, posterior ao treino do modelo? Aí ele precisa **agir**:
buscar a informação.

O Claude tem uma **ferramenta nativa de busca na web** (*server-side*): você apenas a declara,
e o próprio modelo decide *quando* buscar, executa a busca na infraestrutura da Anthropic, lê
os resultados e responde — **já com as citações das fontes**. É exatamente o ciclo **ReAct**
(*Reasoning + Acting*, Yao et al., 2022), só que orquestrado automaticamente:

```
Pensamento: preciso de um dado atual que não tenho.
Ação: web_search[consulta]
Observação: <resultados da web>
Resposta Final: <resposta fundamentada + fontes>
```

> Essa ferramenta existe **apenas nos modelos da Anthropic (Claude)**. Por isso, esta seção
> usa o SDK `anthropic` diretamente (em vez do LangChain das seções anteriores). Requer
> `ANTHROPIC_API_KEY` e acesso à internet.

In [9]:
import os
import anthropic

# A Web Search nativa é uma ferramenta da Anthropic, então aqui usamos o SDK `anthropic`
# diretamente. O cliente lê a ANTHROPIC_API_KEY do ambiente (carregada do .env na célula 1).
cliente = anthropic.Anthropic()
MODELO_CLAUDE = os.getenv("LLM_MODEL") or "claude-sonnet-4-6"

# Ferramenta server-side: nós só declaramos; o Claude decide quando e como usá-la.
# `max_uses` limita quantas buscas ele pode fazer por pergunta.
FERRAMENTA_BUSCA = {"type": "web_search_20250305", "name": "web_search", "max_uses": 4}


def perguntar_com_busca(pergunta, verbose=True):
    """Pergunta ao Claude com a Web Search ativa. Mostra o ciclo ReAct (ação -> observação)
    e devolve (resposta_final, fontes). A busca roda na infraestrutura da Anthropic."""
    mensagens = [{"role": "user", "content": pergunta}]

    for _ in range(5):  # repete enquanto o modelo pedir para continuar (stop_reason pause_turn)
        resp = cliente.messages.create(
            model=MODELO_CLAUDE, max_tokens=1024,
            tools=[FERRAMENTA_BUSCA], messages=mensagens)

        # Mostra o que o modelo fez nesta rodada: as consultas e quantos resultados vieram.
        for bloco in resp.content:
            if bloco.type == "server_tool_use" and bloco.input.get("query"):
                if verbose:
                    print(f"  Ação        -> web_search[{bloco.input['query']}]")
            elif bloco.type == "web_search_tool_result" and not isinstance(bloco.content, dict):
                if verbose:
                    print(f"  Observação  -> {len(bloco.content)} resultados da web")

        if resp.stop_reason == "pause_turn":   # atingiu o limite interno de iterações; continuar
            mensagens.append({"role": "assistant", "content": resp.content})
            continue

        # Resposta final: concatena os blocos de texto e coleta as fontes citadas (título + URL).
        resposta = "".join(b.text for b in resp.content if b.type == "text").strip()
        fontes = []
        for b in resp.content:
            if b.type == "text":
                for cit in (getattr(b, "citations", None) or []):
                    par = (getattr(cit, "title", None), getattr(cit, "url", None))
                    if par[1] and par not in fontes:
                        fontes.append(par)
        return resposta, fontes

    return "(limite de iterações atingido)", []

### Uma pergunta que o modelo não tem como saber

A informação abaixo muda com o tempo e pode ser **posterior ao corte de conhecimento** do
modelo — sem buscar na web, ele não teria como garantir o acerto. Rode e observe o ciclo: o
Claude formula a **consulta**, recebe os **resultados** e só então dá a **resposta final**, com
as **fontes** que embasaram a resposta.

In [10]:
pergunta = "Qual é a versão estável mais recente do Python e em que data ela foi lançada?"
print("Pergunta:", pergunta, "\n")

resposta, fontes = perguntar_com_busca(pergunta)

print("\nResposta Final:\n" + resposta)
print("\nFontes citadas:")
for titulo, url in fontes:
    print(f"  - {titulo}: {url}")

Pergunta: Qual é a versão estável mais recente do Python e em que data ela foi lançada? 

  Ação        -> web_search[Python latest stable version release date 2026]
  Observação  -> 9 resultados da web

Resposta Final:
A versão estável mais recente do Python é a **Python 3.14.8**. O Python 3.14.8 foi lançado em **30 de setembro de 2026**.

A série 3.14 é atualmente a versão em correção de bugs (*bugfix*), com lançamento inicial em outubro de 2025, com suporte previsto até outubro de 2030 (conforme a PEP 745).

Vale destacar que o Python 3.15.0rc1 já está disponível em pré-lançamento, com a versão estável esperada para outubro de 2026.

Fontes citadas:
  - Python documentation by version: https://www.python.org/doc/versions/
  - Download Python: https://www.python.org/downloads/
  - Python (programming language): https://en.wikipedia.org/wiki/Python_(programming_language)


### Combinando: Web Search + structured output

A busca devolve a resposta em **prosa** — ótima para ler, ruim para usar em código. Aplicando
o **structured output** (seção 3) sobre essa resposta, obtemos um **objeto validado**
(`versao`, `data_lancamento`) e ainda anexamos a **fonte** rastreável que a busca citou.

É o pipeline típico de um agente: **agir** (buscar o fato atual) e depois **estruturar**
(entregar um dado pronto para o sistema consumir).

In [11]:
import json
from pydantic import BaseModel, Field


# Etapa 2: transformar a resposta em prosa (da busca) num objeto validado.
class VersaoPython(BaseModel):
    """Dados estruturados extraídos da resposta da busca."""
    versao: str = Field(description="Versão estável mais recente do Python, ex.: '3.14'")
    data_lancamento: str = Field(description="Data de lançamento dessa versão")


# messages.parse força o schema e devolve um objeto Pydantic já validado.
extracao = cliente.messages.parse(
    model=MODELO_CLAUDE, max_tokens=400,
    messages=[{"role": "user", "content": "Extraia os dados deste texto:\n" + resposta}],
    output_format=VersaoPython)

fato = extracao.parsed_output
print("Objeto validado:", fato)
print("  versao         :", fato.versao)
print("  data_lancamento:", fato.data_lancamento)

# Juntando tudo: fato fresco (busca) + estrutura validada + fonte rastreável.
resultado = {"versao": fato.versao,
             "data_lancamento": fato.data_lancamento,
             "fonte": fontes[0][1] if fontes else None}
print("\nResultado final, pronto para uso em código:")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

Objeto validado: versao='3.14.8' data_lancamento='30 de setembro de 2026'
  versao         : 3.14.8
  data_lancamento: 30 de setembro de 2026

Resultado final, pronto para uso em código:
{
  "versao": "3.14.8",
  "data_lancamento": "30 de setembro de 2026",
  "fonte": "https://www.python.org/doc/versions/"
}


## Conclusão

Vimos que a qualidade da saída de um LLM depende muito mais do **prompt** do que parece.
Saindo de pedidos vagos, cada técnica — clareza, *system*, estrutura, *few-shot*,
*chain-of-thought* e aterramento — corrige uma classe de falha. E quando o conhecimento do
modelo não basta, o padrão **ReAct** o conecta a **ferramentas** (como a Web Search nativa),
fundamento dos agentes que colocaremos em produção nas próximas aulas.

**Desafios (opcionais):**
1. Faça uma pergunta que exija **dois fatos** (ex.: "Quem é o atual técnico do Flamengo e quando foi contratado?") e observe o Claude encadear mais de uma busca.
2. Restrinja as buscas a domínios confiáveis com o parâmetro `allowed_domains` na ferramenta (ex.: `"allowed_domains": ["python.org"]`) e compare as fontes citadas.
3. Combine as técnicas: peça a resposta da busca em **JSON** com *structured output* (seção 3), por exemplo `{"versao": "...", "data": "..."}`.